# Lab 20-11 · Mismatch, Gradients and Statistical Design

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import matching

rng = np.random.default_rng(11)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The area law

*Predict* the slope of σ(ΔV_T) against $1/\sqrt{WL}$ for $A_{VT}$ = 3.5 mV·µm.

In [2]:
areas = np.array([0.5, 1, 2, 5, 10, 20, 50])
sig = [matching.mc_pair(3.5, a, 1, 64, 4000, rng).std(ddof=1) for a in areas]
x = 1 / np.sqrt(areas)
slope = np.polyfit(x, sig, 1)
print(f"fitted slope {slope[0]:.2f} mV um, intercept {slope[1]:.3f} mV")
plt.figure()
plt.plot(x, sig, "o", label="Monte Carlo")
plt.plot(x, 3.5 * x, "k--", label="A_VT / sqrt(WL)")
plt.xlabel("1 / sqrt(WL)  (1/um)"); plt.ylabel("sigma(dV_T) (mV)"); plt.legend(); plt.show()

fitted slope 3.46 mV um, intercept 0.012 mV


## Part 2 · Mirrors, pairs and ngspice

*Predict* the current mismatch of the three mirrors in `mirror.cir` (1 mV threshold step at $g_m/I_D$ = 20, 6.32, 3.16 V⁻¹).

In [3]:
svt, sb = 3.5e-3 / math.sqrt(10), 0.01 / math.sqrt(10)
g = np.linspace(3, 25, 200)
plt.figure()
plt.plot(g, [matching.mirror_sigma(k, svt, sb) * 100 for k in g], label="mirror sigma(dI/I) (%)")
plt.plot(g, [matching.pair_offset_sigma(k, svt, sb) * 1e3 for k in g], label="pair sigma(Vos) (mV)")
plt.xlabel("gm/ID (1/V)"); plt.legend(); plt.show()
res = ngspice("mirror.cir")
for name, iref in (("mma", 10e-6), ("mmb", 100e-6), ("mmc", 400e-6)):
    vov = math.sqrt(2 * iref / 2e-3)
    print(f"Iref {iref * 1e6:4.0f} uA: gm/ID {2 / vov:5.2f}/V  formula {2 / vov * 0.1:.3f} %  ngspice {res[name]:.3f} %")

Iref   10 uA: gm/ID 20.00/V  formula 2.000 %  ngspice 1.990 %
Iref  100 uA: gm/ID  6.32/V  formula 0.632 %  ngspice 0.631 %
Iref  400 uA: gm/ID  3.16/V  formula 0.316 %  ngspice 0.316 %


## Part 3 · Gradients and placement

*Predict* which patterns cancel linear and which cancel quadratic gradients.

In [4]:
for pat in ("AB", "AABB", "ABAB", "ABBA", "ABBABAAB"):
    r1, r2 = matching.pattern_residuals(pat)
    off = matching.gradient_offset(pat, [0.01, 1e-4], 10)
    print(f"{pat:9s} linear residual {r1:+.2f} pitch, quadratic {r2:+.2f} pitch^2, offset {off * 1e3:+.1f} uV")

AB        linear residual -1.00 pitch, quadratic +0.00 pitch^2, offset -100.0 uV
AABB      linear residual -2.00 pitch, quadratic +0.00 pitch^2, offset -200.0 uV
ABAB      linear residual -1.00 pitch, quadratic +0.00 pitch^2, offset -100.0 uV
ABBA      linear residual +0.00 pitch, quadratic +2.00 pitch^2, offset +20.0 uV
ABBABAAB  linear residual +0.00 pitch, quadratic +0.00 pitch^2, offset -0.0 uV


## Part 4 · From σ to yield

*Predict* the fraction of offsets outside ±3 mV for σ = 1.107 mV.

In [5]:
vos = rng.normal(0, 1.107, 1_000_000)
print(f"simulated {np.mean(abs(vos) > 3) * 100:.2f} %, formula {2 * stats.norm.sf(3 / 1.107) * 100:.2f} %")

simulated 0.67 %, formula 0.67 %


## Part 5 · How many Monte Carlo runs?

*Predict* how widely a 100-run σ estimate scatters, and how many failure-free runs a 4.5σ claim needs.

In [6]:
est = rng.normal(0, 1, (2000, 100)).std(axis=1, ddof=1)
lo, hi = matching.sigma_ci(100)
print(f"interval {lo:.3f} to {hi:.3f}; fraction of estimates inside: {np.mean((est >= lo) & (est <= hi)):.3f}")
plt.figure()
plt.hist(est, bins=50)
plt.axvline(lo, color="k", ls="--"); plt.axvline(hi, color="k", ls="--")
plt.xlabel("estimated sigma / true sigma"); plt.ylabel("count"); plt.show()
for k in (3, 4, 4.5):
    p = 2 * stats.norm.sf(k)
    print(f"{k} sigma: failure rate {p:.2e}, failure-free runs needed {matching.zero_failure_runs(p)}")

interval 0.878 to 1.162; fraction of estimates inside: 0.946
3 sigma: failure rate 2.70e-03, failure-free runs needed 1109
4 sigma: failure rate 6.33e-05, failure-free runs needed 47293
4.5 sigma: failure rate 6.80e-06, failure-free runs needed 440850
